# Chapter 10: The Full GPT Architecture

[Read this chapter online](https://jackluu.io/book/section-3-the-transformer/ch10-full-gpt-architecture/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch10-full-gpt-architecture.ipynb)

From *Teach Your Computer to Write: Build and Train an LLM from Zero* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 10: The Full GPT Architecture

![The map with Transformer Blocks and Next-Token Scores highlighted](https://jackluu.io/book/assets/diagrams/ch10-where-we-are.png)

*Figure 10.1: We connect everything to output a prediction.*

We have built all the core components of a Transformer model in previous chapters, so we can now assemble them (Figure 10.1). In this chapter you will:

* Build the full GPT model from end to end.
* See how data flows through the entire pipeline.
* Understand the final raw scores, called logits.

> **Words to Know**
>
> - **LM Head**: The model's last step, which turns each token's internal numbers into one score for every character in the vocabulary.
> - **Logits**: The raw scores the model outputs for each possible next token.

## Theory

### The Final Assembly

The complete GPT model is simple once you have the pieces. 

![The pipeline flows from embeddings through blocks to the LM head](https://jackluu.io/book/assets/diagrams/ch10-stacked-architecture.png)

*Figure 10.2: The full forward pass of the GPT model.*

It flows like a production assembly line (Figure 10.2):

1. Turn text into a list of numbers (Token IDs).
2. Look up the meaning and position vectors (Embeddings).
3. Process them through a stack of Transformer blocks.
4. Tidy up the numbers one last time (Final LayerNorm).
5. Map the internal 128 numbers back to the 65 possible characters (LM Head).

### Tracing the Flow

Let's follow a specific example. Imagine we give the model the word `"ROMEO"` (5 tokens).

**Step 1: Input**
The input is `[30, 27, 25, 17, 27]`. Each of those is the token ID of an individual character in "ROMEO". 30 is token 'R', 27 is 'O', 25 is 'M', and 17 is 'E' in our vocabulary.

**Step 2: Embeddings**
The model converts these 5 IDs into 5 rich vectors (128 numbers each), combining both meaning and position.

**Step 3: Transformer Blocks**
That information will be passed through 4 blocks. By the time it passes through all 4 blocks, the token `O` will have some knowledge of every other letter preceding it. Those 128 numbers will encode information such as "I am the final letter of a name in a famous play."

**Step 4 & 5: Final Output**
One more LayerNorm helps stabilize those numbers. And then the **LM Head** (Language Model Head) takes those 128 numbers and transforms them into 65 numbers. Why 65? Because there are 65 unique characters in our Shakespeare dataset.

![The internal 128 numbers are projected to 65 vocabulary scores](https://jackluu.io/book/assets/diagrams/ch10-lm-head.png)

*Figure 10.3: The LM Head projects internal data back to our vocabulary.*

The model predicts 65 scores per token in the sequence (see Figure 10.3). Positions in the sequence independently predict the next character. The token with the largest score in the final position is the predicted character following "ROMEO".

### What Are Logits?

Those 65 numbers that our LM Head gives us are known as **logits**.

A logit is a technical term for "raw score" (see Figure 10.4). They can be negative, zero, or very large. They also don't sum to equal 1 so we cannot call them probabilities. We will turn these raw scores into percentages later in Chapter 11 when we learn how to generate text. But just remember a bigger logit equals a higher chance of that character being chosen.

![Top three logits at position 0 of an untrained model](https://jackluu.io/book/assets/diagrams/ch10-logits.png)

*Figure 10.4: Logits are the raw scores for the next character.*

We have coded everything from Text down to Next-Token Scores on the big picture map in Figure 10.1. Our model is complete, but it just makes random guesses at this point and we must train it.

## Code

```python
class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        # Look up table for token vectors
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        # Look up table for position vectors
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)

        # A sequence of transformer blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock() for _ in range(cfg.n_layers)
        ])

        # Final layer normalization and linear layer to output scores
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape
        assert T <= self.cfg.block_size, \
            f"Sequence length {T} exceeds block_size " \
            f"{self.cfg.block_size}"

        tok_emb = self.token_emb(token_ids)
        positions = torch.arange(T, device=token_ids.device)
        pos_emb = self.pos_emb(positions)

        # Combine token and position embeddings
        x = tok_emb + pos_emb

        # Pass through the transformer blocks
        x = self.blocks(x)

        # Final normalization and produce scores
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits
```

Run the script to see the parameter breakdown and a test run.

```text
$ python src/ch09_gpt_model.py
Model parameter breakdown:
  Token embedding  :      8,320
  Position embedding:    16,384
  4 Transformer blocks:   791,552
  LM head          :      8,320
  LayerNorm (final):        256
  ---
  TOTAL            :    824,832

Input  shape: torch.Size([2, 10])   (B, T)
Output shape: torch.Size([2, 10, 65])  (B, T, vocab_size)

At each position, model outputs 65 scores.
The highest score = best guess for next character.

First position logits (top 5 scores):
  token 26: 1.861
  token 21: 1.195
  token 44: 1.118
  token 57: 1.071
  token 34: 0.972

Note: these are random (untrained model).
```

**What just happened:**

1. Line 1 assembles the complete GPT class.
2. Line 12 actually instantiates the four blocks. These account for 791,552 of the 824,832 parameters (weights) in the entire model. The original GPT-2 Small had 117 million parameters. Same architecture, just smaller.
3. Line 38 finishes the forward pass, where the model outputs 65 logits (scores) for each position.

**Shape Check:**

- Input token IDs: `[Batch, Time]`
- Output logits: `[Batch, Time, 65]` (Vocab size is 65)

## Try It

> **Try It**
>
> Open `src/ch09_gpt_model.py`. Locate the line `config = GPTConfig()` and insert a new line underneath it: `config.n_layers = 6`. Execute the script once more. The parameter count grows, and most of the parameters are in the blocks.
>

> **In Business**
>
> Think of building our final GPT model as building a pipeline out of Lego bricks. You've got your data reader (embeddings), processor (your blocks) and output (the LM head). You can upgrade your house-style GPT's processor by stacking more layers on without having to replace any other parts of your pipeline.

## Key Takeaways

* The full model stacks embeddings, Transformer blocks, and a final LM head.
* The LM head is the last step, which turns each token's internal numbers into one score for every character in the vocabulary.
* The model outputs logits, which are raw scores for the next token and are not probabilities yet.
* Our complete model has ~825K parameters, so this architecture can be built at a small scale.

## Check Your Understanding

1. If the input sequence has 10 tokens, how many predictions does the model make?
2. Why does the LM head output exactly 65 numbers per token?
3. What is a logit?

## Assignment

> **Assignment 10: Shorter text, same model**
>
> **About 15 minutes. You change one number and write no new code.** Work in this chapter's Colab notebook or on your own laptop.
>
> 1. **Run it.** Run `src/ch09_gpt_model.py` as it is. Copy three lines: the `TOTAL` line and the two shape lines, `Input shape` and `Output shape`.
> 2. **Change one number.** Near the bottom of the file, find the line `B, T = 2, 10`. `B` is how many test sequences the script feeds in, and `T` is how many tokens each one holds. Change `10` to `5`, the length of `ROMEO`, and run the script again.
> 3. **Compare.** Which numbers in the two shape lines changed, and which did not? Did the `TOTAL` line change? How many sets of 65 scores does the model now give for each sequence? The five scores at the bottom shift too. They come from an untrained model, so set them aside.
> 4. **Explain it.** In two or three sentences a manager could follow, say what the model hands back when it reads a five-character input such as `ROMEO`, and why a shorter input does not make the model itself any smaller.
>
> **Hand in:** both sets of three lines, your answers to step 3, and your sentences.

## Further Reading

**One model, many jobs, no retraining.** The researchers wanted to know if a model trained exclusively to predict the next word would learn capabilities nobody had trained it for. Because it was trained on such a broad, diverse swath of web pages, it started answering questions, summarizing, and translating with zero task-specific training when given a prompt that made those tasks obvious. That answered the architectural question of how to build a system for text generation in favor of a decoder architecture that predicts the next token. That's the model we cover in Chapter 10.


Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). *Language models are unsupervised multitask learners* [Technical report]. OpenAI. https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

---

### `src/ch09_gpt_model.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch09_gpt_model.py"   # a cell has none, and the file uses it to find the text

"""
Construct the full GPT architecture.
This file belongs to Chapter 10.
Run: python src/ch09_gpt_model.py
"""
import torch
import torch.nn as nn
import os
import sys

from src.utils.config import GPTConfig
from src.ch08_transformer_block import TransformerBlock

# Settings
config = GPTConfig()

# --- The Idea ---
class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg

        # Look up table for token vectors
        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        # Look up table for position vectors
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)

        # A sequence of transformer blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock() for _ in range(cfg.n_layers)
        ])

        # Final layer normalization and linear layer to output scores
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape
        assert T <= self.cfg.block_size, \
            f"Sequence length {T} exceeds block_size " \
            f"{self.cfg.block_size}"

        tok_emb = self.token_emb(token_ids)
        positions = torch.arange(T, device=token_ids.device)
        pos_emb = self.pos_emb(positions)

        # Combine token and position embeddings
        x = tok_emb + pos_emb

        # Pass through the transformer blocks
        x = self.blocks(x)

        # Final normalization and produce scores
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 10: The Full GPT Model\n")

    model = GPT(config)
    total_params = sum(p.numel() for p in model.parameters())

    print("Model parameter breakdown:")
    print(f"  Token embedding  : {model.token_emb.weight.numel():>10,}")
    print(f"  Position embedding: {model.pos_emb.weight.numel():>9,}")
    block_params = sum(p.numel() for p in model.blocks.parameters())
    print(f"  {config.n_layers} Transformer blocks: {block_params:>9,}")
    lm_head_params = sum(p.numel() for p in model.lm_head.parameters())
    print(f"  LM head          : {lm_head_params:>10,}")
    ln_f_params = sum(p.numel() for p in model.ln_f.parameters())
    print(f"  LayerNorm (final): {ln_f_params:>10,}")
    print("  ---")
    print(f"  TOTAL            : {total_params:>10,}")

    B, T = 2, 10
    token_ids = torch.randint(0, config.vocab_size, (B, T))
    logits = model(token_ids)

    print(f"\nInput  shape: {token_ids.shape}   (B, T)")
    print(f"Output shape: {logits.shape}  (B, T, vocab_size)")
    print(f"\nAt each position, model outputs {config.vocab_size} scores.")
    print("The highest score = best guess for next character.")

    first_pos_logits = logits[0, 0]
    print("\nFirst position logits (top 5 scores):")
    top5 = first_pos_logits.topk(5)
    for score, idx in zip(top5.values.tolist(), top5.indices.tolist()):
        print(f"  token {idx:2d}: {score:.3f}")

    print("\nNote: these are random (untrained model).")
    print("\nFull GPT model done! Ready for Chapter 11.")